# AGEMR：按 pooled 主效应 P 分层，比较 1-df 与旧 4-df spline interaction

**分组：** All、P_main < 10⁻³、P_main < 10⁻⁵、P_main < 5×10⁻⁸；这些是嵌套集合。每组使用同一份无交互 pooled GWAS P，比较线性交互 1-df 与旧 spline 交互 4-df 的 nominal / Bonferroni 数量和比例。

**只读取已经保存的结果**，不调用 PLINK、不重新拟合模型、不下载 controlled data。要求已经运行完 linear notebook 和 pooled-main 分层 notebook。可在一个新 Python kernel 中 Run All；默认选择最近一个同时具有上述结果的完整 linear run，打印所选路径供核对，也可手动指定。

**统一阈值：** nominal = P_int < 0.05；Bonferroni = P_int < 0.05/M，M 是原完整面板大小（本项目为 1,902，阈值约 2.6288×10⁻⁵）。不根据分组后 SNP 数放宽阈值。这不是跨全部模型、表型和探索分组统一校正后的显著性。

**公平比较：** 主表只使用 pooled、1-df、4-df 均有效且 OBS_CT 相同的共同 SNP，确保两种模型每组分母相同；另外保留各模型按自身有效集合的汇总，用于复核此前结果。原始匹配/QC 表、入组 SNP 列表和不一致位点均保存。

In [ ]:
from pathlib import Path
import json, hashlib, uuid, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

LINEAR_RUN = None  # 如需指定，填入上一份 linear notebook 打印的完整 run 目录。
B1_ROOT = Path.home() / "workspace/agemr-data/final_cohort_v1/B1_age_varying"
TRAITS = ("BMI", "SBP", "DBP")
ALPHA = 0.05
GROUPS = [("All", None), ("P_main < 1e-3", 1e-3),
          ("P_main < 1e-5", 1e-5), ("P_main < 5e-8", 5e-8)]
KEY = ["#CHROM", "POS", "ID", "REF", "ALT"]
MODELS = (("1-df linear", "linear"), ("4-df spline (old)", "spline"))

def require(ok, message):
    if not ok:
        raise RuntimeError(message)

def required_results(run):
    return ([run / "qc/variant_panel.tsv"] +
            [run / f"{t}_linear_age_all.tsv" for t in TRAITS] +
            [run / "main_effect_stratified_proportions" / f"{t}_pooled_main.tsv" for t in TRAITS])

if LINEAR_RUN is None:
    candidates=[]
    for marker in (B1_ROOT / "B1_linear_age").glob("*/RUN_COMPLETE.json"):
        meta=json.loads(marker.read_text())
        if meta.get("status")=="complete" and all(p.is_file() for p in required_results(marker.parent)):
            candidates.append((meta.get("finished_utc", ""), marker.parent))
    require(bool(candidates), "No completed linear + pooled-main run found. Set LINEAR_RUN explicitly, or finish the pooled-main notebook first.")
    LINEAR_RUN=max(candidates, key=lambda z:z[0])[1]
else:
    LINEAR_RUN=Path(LINEAR_RUN).expanduser()
completion=json.loads((LINEAR_RUN / "RUN_COMPLETE.json").read_text())
require(completion.get("status")=="complete", "Selected linear run is incomplete.")
require(set(TRAITS).issubset(completion["traits"]), "Selected run does not contain all three phenotypes.")
require(all(p.is_file() for p in required_results(LINEAR_RUN)), "Missing saved pooled-main/linear files; run the previous pooled-main notebook first.")
M=int(completion["panel_variants"])
BONF=ALPHA/M
B1_CURRENT=LINEAR_RUN.parent.parent
stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:6]
OUT=LINEAR_RUN / "main_effect_stratified_1df_vs_4df" / stamp
for folder in (OUT, OUT/"qc", OUT/"groups", OUT/"figures"):
    folder.mkdir(parents=True, exist_ok=True)
print("Using existing linear run:", LINEAR_RUN)
print(f"Original panel M={M}; fixed Bonferroni threshold={BONF:.6e}")
print("Comparison outputs:", OUT)

source_records=[]
def digest(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):
            h.update(block)
    return h.hexdigest()

def track(path, role):
    source_records.append({"role":role,"path":str(path),"bytes":Path(path).stat().st_size,"sha256":digest(path)})

def read_keyed(path, role):
    path=Path(path)
    require(path.is_file(),f"Missing {role}: {path}")
    x=pd.read_csv(path,sep="\t",dtype=str,keep_default_na=False)
    require(set(KEY).issubset(x),f"{role}: unexpected schema")
    x["#CHROM"]=x["#CHROM"].str.replace(r"^chr","",regex=True)
    pos=pd.to_numeric(x["POS"],errors="raise")
    require(np.isfinite(pos).all() and pos.eq(np.floor(pos)).all(),f"{role}: invalid POS")
    x["POS"]=pos.astype(np.int64)
    require(not x.duplicated(KEY).any(),f"{role}: duplicate variant keys")
    track(path,role)
    return x

def truth(x):
    return x.astype(str).str.lower().isin(["true","1"])

def num(x, col):
    require(col in x,f"Missing required column {col}")
    return pd.to_numeric(x[col],errors="coerce")

PANEL=read_keyed(LINEAR_RUN / "qc/variant_panel.tsv","original_panel")[KEY]
require(len(PANEL)==M,"Saved panel size differs from completion record")

## 读取 pooled main、1-df 和旧 4-df，并核对入组集合

旧 4-df 优先读取 linear run 中已经保存的 `*_linear_vs_old_4df.tsv` 快照，固定为当时比较的旧结果；若快照缺失，则读原 B1 目录的 `*_age_varying_joint_tests.tsv`。快照存在时沿用其旧模型有效性和参数检查标记。原文件回退会核对线性运行时记录的旧文件 SHA256（若有）。

4-df 使用历史 **USER_4DF 原始 P**，不把 F 值当成 chi-square，也不静默替换为重算 P。1-df 使用 **ADDxAGE10 单项 t-test P**；pooled 使用无交互模型的 ADD P。所有阈值均严格使用 `<`。

主效应分组对两个交互模型完全相同。OBS_CT 相同只核对了人数；历史运行缺少完整输入 hash 时，无法据此证明旧模型个体身份完全相同。本 notebook 保留该限制，不将描述性比例比较解释为正式 enrichment 或模型优劣检验。

In [ ]:
def load_old(trait):
    snapshot=LINEAR_RUN / f"{trait}_linear_vs_old_4df.tsv"
    if snapshot.is_file():
        z=read_keyed(snapshot,f"{trait}_4df_snapshot")
        require({"TEST_4df","P_4df","OBS_CT_4df","ERRCODE_4df"}.issubset(z),f"{trait}: snapshot lacks 4-df columns")
        old=z.loc[z["TEST_4df"].eq("USER_4DF"),KEY].copy()
        idx=old.index
        old["P_spline"]=num(z,"P_4df").loc[idx]
        old["N_spline"]=num(z,"OBS_CT_4df").loc[idx]
        old["ERR_spline"]=z.loc[idx,"ERRCODE_4df"]
        old["A1_spline"]=z.loc[idx,"A1_4df"] if "A1_4df" in z else ""
        old["old_QC_ok"]=True
        if "VALID_4df" in z:
            old["old_QC_ok"] &= truth(z.loc[idx,"VALID_4df"])
        if "F_P_consistent_4df" in z:
            old["old_QC_ok"] &= truth(z.loc[idx,"F_P_consistent_4df"])
        old["old_parameter_status"]=z.loc[idx,"old_log_parameter_status"] if "old_log_parameter_status" in z else "unverified"
        old["old_QC_ok"] &= old["old_parameter_status"].ne("parameters_mismatch_or_unreadable")
        old["historical_input_identity_verified"]=truth(z.loc[idx,"historical_input_identity_verified"]) if "historical_input_identity_verified" in z else False
        old["old_source_kind"]="saved_linear_comparison_snapshot"
    else:
        path=B1_CURRENT / f"{trait}_age_varying_joint_tests.tsv"
        z=read_keyed(path,f"{trait}_4df_original")
        require(z["TEST"].eq("USER_4DF").all(),f"{trait}: old table contains tests other than USER_4DF")
        record_path=LINEAR_RUN / "qc/input_files.json"
        if record_path.is_file():
            records=json.loads(record_path.read_text())
            recorded=next((r for r in records if Path(r["path"]).name==path.name),None)
            if recorded and recorded.get("sha256"):
                require(digest(path)==recorded["sha256"],f"{trait}: old file changed since linear run; cannot silently use it")
        old=z[KEY].copy()
        old["P_spline"]=num(z,"P")
        old["N_spline"]=num(z,"OBS_CT")
        old["ERR_spline"]=z["ERRCODE"]
        old["A1_spline"]=z["A1"] if "A1" in z else ""
        old["old_QC_ok"]=True
        old["old_parameter_status"]="unverified_original_file_fallback"
        old["historical_input_identity_verified"]=False
        old["old_source_kind"]="original_B1_file"
        warnings.warn(f"{trait}: no saved comparison snapshot; using original USER_4DF results. Historical input identity and parameter logs are not fully verified here.")
    old["valid_spline"]=(old["ERR_spline"].eq(".") & old["P_spline"].between(0,1) &
                           old["N_spline"].gt(0) & old["old_QC_ok"])
    return old.reset_index(drop=True)

matched={}
qc_rows=[]
for trait in TRAITS:
    main_path=LINEAR_RUN / "main_effect_stratified_proportions" / f"{trait}_pooled_main.tsv"
    main=read_keyed(main_path,f"{trait}_pooled_main")
    require(main["TEST"].eq("ADD").all(),f"{trait}: main-effect table must contain only ADD")
    a=main[KEY].copy()
    a["P_main"]=num(main,"P")
    a["N_main"]=num(main,"OBS_CT")
    a["ERR_main"]=main["ERRCODE"]
    a["valid_main"]=a["ERR_main"].eq(".") & a["P_main"].between(0,1) & a["N_main"].gt(0)
    linear=read_keyed(LINEAR_RUN / f"{trait}_linear_age_all.tsv",f"{trait}_1df")
    require(linear["TEST"].eq("ADDxAGE10").all(),f"{trait}: expected ADDxAGE10 rows")
    b=linear[KEY].copy()
    b["P_linear"]=num(linear,"P")
    b["N_linear"]=num(linear,"OBS_CT")
    b["ERR_linear"]=linear["ERRCODE"]
    b["A1_linear"]=linear["A1"] if "A1" in linear else ""
    b["valid_linear"]=truth(linear["VALID"]) & b["ERR_linear"].eq(".") & b["P_linear"].between(0,1) & b["N_linear"].gt(0)
    if "rsid" in linear:
        b["rsid"]=linear["rsid"]
    old=load_old(trait)
    # Audit all keys, including extras. Primary comparison always starts from the saved original panel.
    for source,label in ((a,"main"),(b,"linear"),(old,"spline")):
        coverage=PANEL.merge(source[KEY],on=KEY,how="outer",validate="one_to_one",indicator=True)
        coverage.to_csv(OUT / "qc" / f"{trait}_{label}_key_coverage.tsv",sep="\t",index=False)
        qc_rows.append({"trait":trait,"source":label,"N_source":len(source),
                        "N_missing_from_panel":int(coverage["_merge"].eq("left_only").sum()),
                        "N_extra_outside_panel":int(coverage["_merge"].eq("right_only").sum())})
    x=PANEL.merge(a,on=KEY,how="left",validate="one_to_one").merge(b,on=KEY,how="left",validate="one_to_one").merge(old,on=KEY,how="left",validate="one_to_one")
    for col in ("valid_main","valid_linear","valid_spline"):
        x[col]=truth(x[col])
    x["same_N_main_linear"]=x["N_main"].eq(x["N_linear"])
    x["same_N_main_spline"]=x["N_main"].eq(x["N_spline"])
    x["same_A1_models"]=x["A1_linear"].eq(x["A1_spline"])
    x["eligible_linear"]=x["valid_main"] & x["valid_linear"] & x["same_N_main_linear"]
    x["eligible_spline"]=x["valid_main"] & x["valid_spline"] & x["same_N_main_spline"]
    x["eligible_common"]=x["eligible_linear"] & x["eligible_spline"]
    require(x["eligible_common"].any(),f"{trait}: no comparable SNPs; inspect QC")
    for suffix in ("linear","spline"):
        x[f"nominal_{suffix}"]=x[f"P_{suffix}"].lt(ALPHA) & x[f"valid_{suffix}"]
        x[f"Bonf_{suffix}"]=x[f"P_{suffix}"].lt(BONF) & x[f"valid_{suffix}"]
    x["trait"]=trait
    x.to_csv(OUT / f"{trait}_matched_Pmain_1df_4df.tsv",sep="\t",index=False)
    x.loc[~x["eligible_common"]].to_csv(OUT / "qc" / f"{trait}_excluded_from_common.tsv",sep="\t",index=False)
    matched[trait]=x
coverage_summary=pd.DataFrame(qc_rows)
coverage_summary.to_csv(OUT / "qc/key_coverage_summary.tsv",sep="\t",index=False)
display(coverage_summary)

## 分层汇总及逐项对比

主表给出共同集合上的分子/分母、百分比、相对该模型 All 组的比例倍数，以及 `4-df − 1-df` 的百分点差值。另存各模型自己的有效集合，防止共同集合排除部分位点后与此前截图数字不一致却无法定位。

`N_selected_by_main_P` 为完整原面板中符合该主效应 P 条件的位点数；`N_comparable` 才是公平比较的比例分母。空组比例为 NaN；基线为 0 时，相对倍数为 NaN。分组和模型间的差值均为描述性结果。

In [ ]:
def selected_mask(x,cut):
    if cut is None:
        return pd.Series(True,index=x.index)
    return x["P_main"].between(0,1) & x["P_main"].lt(cut)

def make_summary(x,trait):
    rows=[]
    for population in ("common","model_specific"):
        for model,suffix in MODELS:
            ok=x["eligible_common"] if population=="common" else x[f"eligible_{suffix}"]
            reference=x.loc[ok]
            base_nom=reference[f"nominal_{suffix}"].mean() if len(reference) else np.nan
            base_bonf=reference[f"Bonf_{suffix}"].mean() if len(reference) else np.nan
            for label,cut in GROUPS:
                selected=selected_mask(x,cut)
                g=x.loc[selected & ok]
                n=len(g)
                n_nom=int(g[f"nominal_{suffix}"].sum())
                n_bonf=int(g[f"Bonf_{suffix}"].sum())
                r_nom=n_nom/n if n else np.nan
                r_bonf=n_bonf/n if n else np.nan
                rows.append({"trait":trait,"group":label,"model":model,"population":population,
                    "N_selected_by_main_P":int(selected.sum()),"N_comparable":n,
                    "N_excluded_QC":int((selected & ~ok).sum()),
                    "N_nominal":n_nom,"pct_nominal":100*r_nom,
                    "N_Bonf":n_bonf,"pct_Bonf":100*r_bonf,
                    "fold_nominal_vs_All":r_nom/base_nom if base_nom>0 else np.nan,
                    "fold_Bonf_vs_All":r_bonf/base_bonf if base_bonf>0 else np.nan,
                    "fixed_Bonf_threshold":BONF})
    return pd.DataFrame(rows)

summary_long=pd.concat([make_summary(matched[t],t) for t in TRAITS],ignore_index=True)
summary_long.to_csv(OUT / "stratified_1df_4df_long.tsv",sep="\t",index=False)
common_summary=summary_long.loc[summary_long["population"].eq("common")].copy()
common_summary.to_csv(OUT / "stratified_1df_4df_common.tsv",sep="\t",index=False)
model_specific=summary_long.loc[summary_long["population"].eq("model_specific")].copy()
model_specific.to_csv(OUT / "qc/stratified_model_specific.tsv",sep="\t",index=False)

join_cols=["trait","group","N_selected_by_main_P","N_comparable","N_excluded_QC","fixed_Bonf_threshold"]
metrics=["N_nominal","pct_nominal","N_Bonf","pct_Bonf","fold_nominal_vs_All","fold_Bonf_vs_All"]
one=common_summary.loc[common_summary["model"].eq("1-df linear"),join_cols+metrics].rename(columns={c:c+"_1df" for c in metrics})
four=common_summary.loc[common_summary["model"].eq("4-df spline (old)"),join_cols+metrics].rename(columns={c:c+"_4df" for c in metrics})
comparison=one.merge(four,on=join_cols,how="outer",validate="one_to_one",indicator=True)
require(comparison["_merge"].eq("both").all(),"The two model denominators differ unexpectedly")
comparison=comparison.drop(columns="_merge")
comparison["diff_nominal_pp_4df_minus_1df"]=comparison["pct_nominal_4df"]-comparison["pct_nominal_1df"]
comparison["diff_Bonf_pp_4df_minus_1df"]=comparison["pct_Bonf_4df"]-comparison["pct_Bonf_1df"]
comparison.to_csv(OUT / "stratified_1df_vs_4df_side_by_side.tsv",sep="\t",index=False)

def display_table(x):
    show=x.copy()
    for c in show.columns:
        if c.startswith(("pct_","fold_","diff_")):
            show[c]=show[c].round(3)
    if "fixed_Bonf_threshold" in show:
        show["fixed_Bonf_threshold"]=show["fixed_Bonf_threshold"].map(lambda v:f"{v:.4e}")
    display(show)

compact_columns=["trait","group","N_selected_by_main_P","N_comparable","N_excluded_QC",
                 "N_nominal_1df","pct_nominal_1df","N_nominal_4df","pct_nominal_4df",
                 "N_Bonf_1df","pct_Bonf_1df","N_Bonf_4df","pct_Bonf_4df"]
for trait in TRAITS:
    print("\n",trait,": common SNP denominator; fixed Bonferroni =",f"{BONF:.4e}")
    display_table(comparison.loc[comparison["trait"].eq(trait),compact_columns])
print("Relative proportions vs each model's All group:")
display_table(common_summary[["trait","group","model","N_comparable","fold_nominal_vs_All","fold_Bonf_vs_All"]])

In [ ]:
overlap_rows=[]
for trait in TRAITS:
    x=matched[trait]
    for group_index,(label,cut) in enumerate(GROUPS):
        g=x.loc[selected_mask(x,cut) & x["eligible_common"]].copy()
        tag=("All","Pmain_lt_1e3","Pmain_lt_1e5","Pmain_lt_5e8")[group_index]
        for metric in ("nominal","Bonf"):
            one=g[f"{metric}_linear"]
            four=g[f"{metric}_spline"]
            g[f"{metric}_overlap"]=np.select([one & four,one & ~four,~one & four],
                                                  ["both","1df_only","4df_only"],default="neither")
            counts=g[f"{metric}_overlap"].value_counts()
            overlap_rows.append({"trait":trait,"group":label,"criterion":metric,"N_comparable":len(g),
                                 **{s:int(counts.get(s,0)) for s in ("both","1df_only","4df_only","neither")}})
        g.to_csv(OUT / "groups" / f"{trait}_{tag}_paired_variants.tsv",sep="\t",index=False)
        g.loc[g["nominal_linear"].ne(g["nominal_spline"]) | g["Bonf_linear"].ne(g["Bonf_spline"])] .to_csv(
            OUT / "groups" / f"{trait}_{tag}_discordant_variants.tsv",sep="\t",index=False)
overlap=pd.DataFrame(overlap_rows)
overlap.to_csv(OUT / "stratified_significance_overlap.tsv",sep="\t",index=False)
display(overlap)

## 图形：相同分组、相同分母下的两模型显著比例

每个 trait 保存 nominal 与 Bonferroni 两张面板，共享主效应分组顺序。横轴同时标出可比较 SNP 数；没有合格 SNP 的组留空，不画成 0%。线段仅帮助比较嵌套组，不代表连续剂量或独立的重复试验。

In [ ]:
for trait in TRAITS:
    z=comparison.loc[comparison["trait"].eq(trait)].set_index("group").loc[[g[0] for g in GROUPS]]
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    positions=np.arange(len(GROUPS))
    labels=[f"{name}\n(n={int(n)})" for name,n in zip(z.index,z["N_comparable"])]
    for ax,metric,title in zip(axes,["nominal","Bonf"],["Interaction P < 0.05",f"Interaction P < {BONF:.3e}"]):
        for suffix,color,model in [("1df","#24628f","1-df linear"),("4df","#d17a35","4-df spline (old)")]:
            ax.plot(positions,z[f"pct_{metric}_{suffix}"].to_numpy(dtype=float),marker="o",lw=1.7,
                    color=color,label=model)
        ax.set_xticks(positions,labels)
        ax.set(ylabel="Interaction-significant SNPs (%)",title=title)
        ax.grid(axis="y",alpha=0.2)
        ax.legend(fontsize=9)
    fig.suptitle(f"{trait}: main-effect strata; matched variants and sample counts",fontsize=12)
    fig.tight_layout()
    fig.savefig(OUT / "figures" / f"{trait}_stratified_1df_vs_4df.png",dpi=200,bbox_inches="tight")
    fig.savefig(OUT / "figures" / f"{trait}_stratified_1df_vs_4df.pdf",bbox_inches="tight")
    plt.show()
    plt.close(fig)

source_manifest=pd.DataFrame(source_records)
source_manifest.to_csv(OUT / "qc/source_files.tsv",sep="\t",index=False)
result_manifest={"status":"complete","linear_run":str(LINEAR_RUN),"output":str(OUT),
                 "created_utc":datetime.now(timezone.utc).isoformat(),"panel_variants":M,
                 "fixed_Bonf_threshold":BONF,"groups":GROUPS,
                 "primary_denominator":"common valid SNPs; equal OBS_CT across main/1df/4df",
                 "old_P_policy":"historical USER_4DF raw P; no silent recalculation",
                 "new_P_policy":"ADDxAGE10 single-term t-test P",
                 "historical_sample_identity_caveat":True}
(OUT / "COMPARISON_COMPLETE.json").write_text(json.dumps(result_manifest,indent=2,ensure_ascii=False)+"\n")
print("Complete; saved tables and figures to:",OUT)

## 输出与解释

- `stratified_1df_vs_4df_side_by_side.tsv`：主表，每个 trait × 主效应组并排显示两个模型的分子、分母、百分比、相对 All 的倍数和百分点差。
- `stratified_1df_4df_long.tsv`：共同集合与模型各自有效集合的完整长表。
- `stratified_significance_overlap.tsv`、`groups/`：每组两个模型都显著、仅 1-df、仅 4-df、都不显著的数量及对应 SNP 列表。
- `qc/`：匹配覆盖、共同集合排除位点、各模型自身集合、来源文件 hash。
- `figures/`：BMI、SBP、DBP 各一张 PNG 和 PDF。

**解释边界：** 旧 4-df omnibus 与新 1-df linear 检验不同的交互假设；命中比例差异不能直接检验非线性，也不是模型优劣的正式 P 值。主效应分组来自同一样本的 pooled GWAS，分组之间嵌套，比例和倍数均是描述性结果。子集很小时需同时看分子和分母；没有显著 SNP 不等于没有年龄交互。

**验证：** 本文件使用已有 3-trait、22-chromosome 的合成 workspace 测试；验证 1e-3 新分组、严格阈值、共同分母、计数重叠、空子集、零基线和不一致样本数排除。未在用户的真实 controlled cohort 上执行，运行前不推断新的实际结果。